<a href="https://colab.research.google.com/github/HarryWarre/trading-model-ai-lab/blob/main/colab/run_research043_multiyear_confirmation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research 043 — dựng panel và xác nhận FOMC 2023–2025
Một lần chạy: kiểm tra 45 ZIP → dựng bảng rộng 15 tài sản → khóa hash → chạy kiểm thử → chạy mô hình FOMC đã đăng ký trước. Không thay dữ liệu thiếu và không tự chọn tham số.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, pathlib, subprocess
REPO = 'HarryWarre/trading-model-ai-lab'
WORK = pathlib.Path('/content/trading-model-ai-lab')
if not WORK.exists():
    subprocess.run(['git','clone',f'https://github.com/{REPO}.git',str(WORK)],check=True)
else:
    subprocess.run(['git','-C',str(WORK),'pull','--ff-only'],check=True)
os.chdir(WORK)
subprocess.run(['pip','install','-q','pandas','numpy','pytest'],check=True)
print('Repository commit:', subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
from pathlib import Path
import json, subprocess
ROOT = Path('/content/drive/MyDrive/trading-model-ai-lab')
DATA = ROOT / 'data'
RAW = DATA / 'histdata_raw'
ORIGINAL_MANIFEST = RAW / 'histdata_multiyear_manifest.csv'
REVALIDATED_MANIFEST = RAW / 'histdata_multiyear_manifest_revalidated.csv'
PANEL = DATA / 'histdata_m1_5m_15_2023_2025.csv'
PANEL_MANIFEST = PANEL.with_suffix('.manifest.json')
VIX = DATA / 'fred_VIXCLS.csv'
RESULTS = ROOT / 'results' / 'research043'
missing=[str(x) for x in [RAW,ORIGINAL_MANIFEST,VIX] if not x.exists()]
if missing: raise FileNotFoundError('Missing Drive inputs: '+str(missing))
cmd=['python','colab/revalidate_histdata_manifest.py','--original',str(ORIGINAL_MANIFEST),'--raw-dir',str(RAW),'--output',str(REVALIDATED_MANIFEST)]
subprocess.run(cmd,check=True)
revalidation=json.loads(REVALIDATED_MANIFEST.with_suffix('.summary.json').read_text())
assert revalidation['rows']==45 and revalidation['assets']==15
print(json.dumps(revalidation,indent=2))


In [ ]:
import subprocess
subprocess.run(['pytest','-q','test_revalidate_histdata_manifest.py','test_prepare_histdata_m1_panel.py','test_real_fomc_drift_2023_2025.py'],check=True)


In [ ]:
import subprocess
cmd=['python','colab/prepare_histdata_m1_panel.py','--raw-dir',str(RAW),'--download-manifest',str(REVALIDATED_MANIFEST),'--years','2023,2024,2025','--output',str(PANEL),'--min-assets','15']
print('Building immutable wide panel from revalidated manifest...')
subprocess.run(cmd,check=True)


In [ ]:
import json, hashlib, pandas as pd
meta=json.loads(PANEL_MANIFEST.read_text())
assert meta['output']['format']=='wide_close'
assert meta['output']['asset_count']==15
assert meta['output']['years_requested']==[2023,2024,2025]
assert len(meta['coverage'])==45
header=pd.read_csv(PANEL,nrows=5)
assert len(header.columns)==16 and header.columns[0]=='timestamp'
print(json.dumps(meta['output'],indent=2))
print('Panel QA PASS')


In [ ]:
RESULTS.mkdir(parents=True,exist_ok=True)
cmd=['python','real_fomc_drift_2023_2025.py','--panel',str(PANEL),'--panel-manifest',str(PANEL_MANIFEST),'--events','data/fomc_2023_2025_official.csv','--vix',str(VIX),'--output-dir',str(RESULTS)]
print('Running frozen Research 043 confirmation...')
subprocess.run(cmd,check=True)
summary=json.loads((RESULTS/'research043_summary.json').read_text())
print(json.dumps(summary,indent=2))
print('DONE_RESEARCH043')


## Kết quả
Khi thấy `DONE_RESEARCH043`, panel, manifest, coverage và toàn bộ kết quả đã nằm trong Google Drive. Không cần chạy thêm notebook nào cho Research 043.